# Build `silver.nyc_tlc.trips` with PySpark

This notebook reads raw NYC TLC Parquet files from the Bronze layer and publishes a canonical Apache Iceberg table in the Silver Polaris catalog.

## Target architecture

- Bronze raw files
  - `s3a://bronze/mobility/nyc_tlc/yellow/`
  - `s3a://bronze/mobility/nyc_tlc/green/`
- Silver catalog
  - logical catalog: `silver`
  - physical storage root: `s3://silver/`
- Gold catalog
  - logical catalog: `gold`
  - physical storage root: `s3://gold/`

## Published Silver table

- `silver.nyc_tlc.trips`

## Why the notebook is structured this way

This notebook follows a Bronze -> Silver -> Gold medallion pattern: Bronze keeps source-shaped raw data, Silver standardizes and validates it into a canonical business table, and Gold derives curated analytical marts from Silver.

For the table layer, the notebook uses Apache Iceberg with Spark catalog integration and SQL extensions, and connects to Apache Polaris through the Iceberg REST catalog API.

For the raw-file layer, the notebook reads Bronze Parquet files with S3A.

## What this notebook does

1. Defines all runtime values directly in one hardcoded configuration cell
2. Starts a Spark session configured for:
   - Bronze raw reads through `s3a://`
   - Iceberg writes through the Polaris REST catalog
3. Reads Yellow and Green Bronze datasets
4. Validates required source columns before transformation
5. Normalizes both datasets into a single canonical schema
6. Applies baseline Silver data-quality filters
7. Creates the namespace `silver.nyc_tlc` if needed
8. Creates or replaces `silver.nyc_tlc.trips`
9. Runs validation queries against the published Iceberg table

## Important runtime note

This notebook sets Spark packages and catalog settings in `SparkSession.builder`. Run it in a fresh kernel so Spark starts with the intended configuration.

The notebook is intentionally kept simple because writes may fail on SeaweedFS with CreateMultipartUpload. This is a SeaweedFS issue on the storage write path, not a problem with the notebook logic. Please, check the [failing notebook](./01_Issue-seaweedFS-build_trips.ipynb).


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F


## Hardcoded runtime configuration

This notebook intentionally uses no environment variables and no mounted-secret file lookups.

All values are hardcoded below. Replace the `REPLACE_...` literals with your real values before running.

### Important Polaris note

The Spark catalog name here is `silver`, and the notebook uses that same logical catalog name in the Iceberg REST configuration.

In this architecture, that logical catalog maps to `s3://silver/`.


## Start Spark

Configuration choices used here:

- `org.apache.iceberg.spark.SparkCatalog` for Iceberg catalog integration in Spark
- Iceberg REST settings pointing to Polaris
- `header.X-Iceberg-Access-Delegation=vended-credentials` so Polaris can delegate storage access for Iceberg-managed tables
- global `s3a` settings for Bronze raw Parquet reads
- Iceberg Spark SQL extensions enabled explicitly

Spark catalogs are configured under `spark.sql.catalog.<catalog_name>`.


In [ ]:
polaris_credential = "my-polaris-spark-etl-app:mySparkAppSecret"

CATALOG_NAME = "silver"
NAMESPACE = "nyc_tlc"
FULL_TABLE_NAME = f"{CATALOG_NAME}.{NAMESPACE}.trips"

spark = (
    SparkSession.builder
    .appName("NYC Tripdata - Silver - PySpark")
    .config(f"spark.sql.catalog.{CATALOG_NAME}.credential", polaris_credential)
    .config("spark.executor.memory", "2g")
    .config("spark.executor.memoryOverhead", "640m")
    .config("spark.executor.cores", 1)
    .config("spark.kubernetes.executor.request.cores", 1)
    .config("spark.kubernetes.executor.limit.cores", 1)
    .getOrCreate()
)


In [ ]:
print("Spark version:", spark.version)
spark.sql("USE silver")
spark.sql("SHOW CATALOGS").show(truncate=False)


In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {FULL_TABLE_NAME} PURGE")

## Read Bronze raw Parquet datasets

Bronze remains source-shaped. Yellow and Green are read independently and normalized only after the source reads succeed.

Spark reads the Bronze Parquet files directly into DataFrames.


In [ ]:
yellow_raw = spark.read.parquet("s3a://bronze/mobility/nyc_tlc/yellow")
green_raw = spark.read.parquet("s3a://bronze/mobility/nyc_tlc/green")

print("Yellow schema")
yellow_raw.printSchema()

print("Green schema")
green_raw.printSchema()

print("Yellow rows:", yellow_raw.count())
print("Green rows:", green_raw.count())


## Validate source schemas before transformation

A sandbox-oriented Silver notebook should fail clearly when required source fields are absent, and it should tolerate known schema differences between Yellow and Green where possible.


In [ ]:
def first_existing_column(df, candidates):
    lower_map = {c.lower(): c for c in df.columns}
    for candidate in candidates:
        real_name = lower_map.get(candidate.lower())
        if real_name is not None:
            return F.col(real_name)
    return None

def require_any_column(df, candidates, dataset_name):
    col_expr = first_existing_column(df, candidates)
    if col_expr is None:
        raise ValueError(
            f"{dataset_name} is missing all candidate columns for {candidates}. "
            f"Available columns: {df.columns}"
        )
    return col_expr

def optional_column(df, candidates, target_type):
    col_expr = first_existing_column(df, candidates)
    if col_expr is None:
        return F.lit(None).cast(target_type)
    return col_expr.cast(target_type)

def required_column(df, candidates, dataset_name, target_type):
    return require_any_column(df, candidates, dataset_name).cast(target_type)


## Canonical mapping

This notebook builds a single trip-level canonical schema from Yellow and Green.

Key design rules:

- keep `source_dataset` for lineage
- preserve the Bronze partition indicator as `source_month`
- normalize pickup and dropoff timestamps and location IDs into common names
- tolerate source-specific fields with nullable columns when they do not exist in the other dataset
- derive `trip_duration_minutes` and `trip_date`
- stamp records with `ingestion_ts`

Spark's `unionByName` combines DataFrames by matching columns by name rather than by position, which is the safer pattern for combining independently normalized source datasets.


In [ ]:
yellow = (
    yellow_raw.select(
        F.lit("yellow").alias("source_dataset"),
        optional_column(yellow_raw, ["month"], "string").alias("source_month"),
        required_column(yellow_raw, ["VendorID", "vendorid"], "yellow", "int").alias("vendor_id"),
        required_column(yellow_raw, ["tpep_pickup_datetime"], "yellow", "timestamp").alias("pickup_ts"),
        required_column(yellow_raw, ["tpep_dropoff_datetime"], "yellow", "timestamp").alias("dropoff_ts"),
        required_column(yellow_raw, ["PULocationID", "pulocationid"], "yellow", "int").alias("pickup_location_id"),
        required_column(yellow_raw, ["DOLocationID", "dolocationid"], "yellow", "int").alias("dropoff_location_id"),
        optional_column(yellow_raw, ["passenger_count"], "double").cast("int").alias("passenger_count"),
        optional_column(yellow_raw, ["trip_distance"], "double").alias("trip_distance"),
        optional_column(yellow_raw, ["RatecodeID", "ratecodeid"], "int").alias("rate_code_id"),
        optional_column(yellow_raw, ["store_and_fwd_flag"], "string").alias("store_and_fwd_flag"),
        optional_column(yellow_raw, ["payment_type"], "int").alias("payment_type"),
        F.lit(None).cast("int").alias("trip_type"),
        optional_column(yellow_raw, ["fare_amount"], "double").alias("fare_amount"),
        optional_column(yellow_raw, ["extra"], "double").alias("extra"),
        optional_column(yellow_raw, ["mta_tax"], "double").alias("mta_tax"),
        optional_column(yellow_raw, ["tip_amount"], "double").alias("tip_amount"),
        optional_column(yellow_raw, ["tolls_amount"], "double").alias("tolls_amount"),
        optional_column(yellow_raw, ["improvement_surcharge"], "double").alias("improvement_surcharge"),
        optional_column(yellow_raw, ["total_amount"], "double").alias("total_amount"),
        optional_column(yellow_raw, ["congestion_surcharge"], "double").alias("congestion_surcharge"),
        optional_column(yellow_raw, ["airport_fee"], "double").alias("airport_fee"),
        optional_column(yellow_raw, ["cbd_congestion_fee"], "double").alias("cbd_congestion_fee"),
    )
)

green = (
    green_raw.select(
        F.lit("green").alias("source_dataset"),
        optional_column(green_raw, ["month"], "string").alias("source_month"),
        required_column(green_raw, ["VendorID", "vendorid"], "green", "int").alias("vendor_id"),
        required_column(green_raw, ["lpep_pickup_datetime"], "green", "timestamp").alias("pickup_ts"),
        required_column(green_raw, ["lpep_dropoff_datetime"], "green", "timestamp").alias("dropoff_ts"),
        required_column(green_raw, ["PULocationID", "pulocationid"], "green", "int").alias("pickup_location_id"),
        required_column(green_raw, ["DOLocationID", "dolocationid"], "green", "int").alias("dropoff_location_id"),
        optional_column(green_raw, ["passenger_count"], "double").cast("int").alias("passenger_count"),
        optional_column(green_raw, ["trip_distance"], "double").alias("trip_distance"),
        optional_column(green_raw, ["RatecodeID", "ratecodeid"], "int").alias("rate_code_id"),
        optional_column(green_raw, ["store_and_fwd_flag"], "string").alias("store_and_fwd_flag"),
        optional_column(green_raw, ["payment_type"], "int").alias("payment_type"),
        optional_column(green_raw, ["trip_type"], "int").alias("trip_type"),
        optional_column(green_raw, ["fare_amount"], "double").alias("fare_amount"),
        optional_column(green_raw, ["extra"], "double").alias("extra"),
        optional_column(green_raw, ["mta_tax"], "double").alias("mta_tax"),
        optional_column(green_raw, ["tip_amount"], "double").alias("tip_amount"),
        optional_column(green_raw, ["tolls_amount"], "double").alias("tolls_amount"),
        optional_column(green_raw, ["improvement_surcharge"], "double").alias("improvement_surcharge"),
        optional_column(green_raw, ["total_amount"], "double").alias("total_amount"),
        optional_column(green_raw, ["congestion_surcharge"], "double").alias("congestion_surcharge"),
        F.lit(None).cast("double").alias("airport_fee"),
        optional_column(green_raw, ["cbd_congestion_fee"], "double").alias("cbd_congestion_fee"),
    )
)

trips_stage = (
    yellow.unionByName(green)
    .where(F.col("pickup_ts").isNotNull() & F.col("dropoff_ts").isNotNull())
    .withColumn(
        "trip_duration_minutes",
        (F.col("dropoff_ts").cast("long") - F.col("pickup_ts").cast("long")) / F.lit(60.0),
    )
    .where(F.col("trip_duration_minutes") >= 0)
    .withColumn("trip_date", F.to_date("pickup_ts"))
    .withColumn("ingestion_ts", F.current_timestamp())
)

trips_stage.printSchema()
trips_stage.show(5, truncate=False)


## Baseline Silver quality checks

Silver should contain validated, canonical data rather than untouched source files. These checks are intentionally simple and operationally useful:

- both datasets should appear in the staged result
- timestamp normalization should look correct
- row counts should be reasonable by source and month
- obvious negative-duration records should already be excluded by the transformation step


In [ ]:
trips_stage.groupBy("source_dataset", "source_month").count().orderBy("source_dataset", "source_month").show(200, truncate=False)

trips_stage.select(
    "source_dataset",
    "pickup_ts",
    "dropoff_ts",
    "pickup_location_id",
    "dropoff_location_id",
    "trip_distance",
    "fare_amount",
    "total_amount",
    "trip_duration_minutes",
).show(20, truncate=False)


## Create namespace and publish the Iceberg table

This section:

1. creates `silver.nyc_tlc` if it does not already exist
2. publishes `silver.nyc_tlc.trips` as an Iceberg table

The table is partitioned by `day(pickup_ts)`.

In [ ]:
spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {CATALOG_NAME}.{NAMESPACE}")

trips_stage.createOrReplaceTempView("trips_stage")

spark.sql(f"""
CREATE OR REPLACE TABLE {FULL_TABLE_NAME}
USING iceberg
PARTITIONED BY (day(pickup_ts))
TBLPROPERTIES (
  'format-version' = '2'
)
AS
SELECT
  source_dataset,
  source_month,
  vendor_id,
  pickup_ts,
  dropoff_ts,
  pickup_location_id,
  dropoff_location_id,
  passenger_count,
  trip_distance,
  rate_code_id,
  store_and_fwd_flag,
  payment_type,
  trip_type,
  fare_amount,
  extra,
  mta_tax,
  tip_amount,
  tolls_amount,
  improvement_surcharge,
  total_amount,
  congestion_surcharge,
  airport_fee,
  cbd_congestion_fee,
  trip_duration_minutes,
  trip_date,
  ingestion_ts
FROM trips_stage
""")

print(f"Published table: {FULL_TABLE_NAME}")


## Validate the published Silver table

These checks confirm that the table exists in the Polaris-backed Silver catalog and contains the expected data.


In [ ]:
spark.sql(f"SHOW TABLES IN {CATALOG_NAME}.{NAMESPACE}").show(truncate=False)
spark.sql(f"DESCRIBE TABLE {FULL_TABLE_NAME}").show(200, truncate=False)

spark.sql(f'''
SELECT
  source_dataset,
  source_month,
  COUNT(*) AS trip_count
FROM {FULL_TABLE_NAME}
GROUP BY source_dataset, source_month
ORDER BY source_dataset, source_month
''').show(200, truncate=False)

spark.sql(f"SELECT * FROM {FULL_TABLE_NAME} LIMIT 20").show(truncate=False)


## Optional operational checks

If your engine and permissions allow it, Iceberg metadata tables can be useful during validation and troubleshooting.


In [ ]:
try:
    spark.sql(f"SELECT * FROM {FULL_TABLE_NAME}.snapshots").show(truncate=False)
except Exception as exc:
    print("Snapshot metadata query skipped:", exc)


In [ ]:
spark.stop()

## What comes next

Downstream Gold notebooks should read only from:

- `silver.nyc_tlc.trips`

and publish:

- `gold.nyc_tlc.monthly_stats`
- `gold.nyc_tlc.zone_demand_daily`
- `gold.nyc_tlc.revenue_monthly`

That keeps the Bronze -> Silver -> Gold lineage clean, reproducible, and aligned with medallion-style lakehouse design.
